In [1]:
import warnings

warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import os
import gc
from tqdm import tqdm
from glob import glob

try:
    import cv2
except Exception:  # pragma: no cover
    cv2 = None

    def _fallback_resize(img, size, interpolation=None):
        return np.resize(img, (size[1], size[0]))

    def _fallback_threshold(img, thresh, maxval, type_):
        return (img > 127).astype(np.uint8) * maxval

    class _FallbackCV2:
        INTER_NEAREST = None
        INTER_LINEAR = None
        THRESH_BINARY = None
        THRESH_OTSU = None
        MORPH_OPEN = None

        @staticmethod
        def resize(img, size, interpolation=None):
            return _fallback_resize(img, size, interpolation)

        @staticmethod
        def threshold(img, thresh, maxval, flags):
            return _fallback_threshold(img, thresh, maxval, flags)

        @staticmethod
        def adaptiveThreshold(
            img, maxValue, adaptiveMethod, thresholdType, blockSize, C
        ):
            return _fallback_threshold(img, 127, maxValue, None)

        @staticmethod
        def morphologyEx(img, op, kernel):
            return img

    cv2 = _FallbackCV2()

tf = None
keras = None
K = None
load_model = None

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras.models import load_model

    K = tf.keras.backend
except BaseException:
    tf = None
    keras = None
    K = None
    load_model = None


def binary_crossentropy(y_true, y_pred):
    """Fallback binary cross‑entropy when TensorFlow is absent."""
    return np.mean(
        -(y_true * np.log(y_pred + 1e-7) + (1 - y_true) * np.log(1 - y_pred + 1e-7))
    )




2026-05-03 16:58:49.034507: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-03 16:58:49.046255: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777827529.060531      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777827529.064671      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-03 16:58:49.079949: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
BATCH_SIZE = 16
EPOCHS = 30
n_splits = 5
fold_selected = 1  # 1..5




In [3]:
def _find_file(rel_path):
    """Search common Kaggle input locations for a relative path."""
    candidates = [
        os.path.join("..", "input", rel_path),
        os.path.join("/kaggle", "input", rel_path),
        os.path.join(".", rel_path),
        rel_path,
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"Unable to locate {rel_path} in any known location.")


# Load sample submission (used as a template for test rows)
sample_sub_path = _find_file(
    "uw-madison-gi-tract-image-segmentation/sample_submission.csv"
)
df = pd.read_csv(sample_sub_path)

DEBUG = False
if df.shape[0] == 0:
    DEBUG = True
if DEBUG:
    train_path = _find_file("uw-madison-gi-tract-image-segmentation/train.csv")
    df = pd.read_csv(train_path)
    df.pop("segmentation")
    df["predicted"] = ""

df.rename(columns={"class": "class_name"}, inplace=True)
df["case"] = df["id"].apply(lambda x: int(x.split("_")[0].replace("case", "")))
df["day"] = df["id"].apply(lambda x: int(x.split("_")[1].replace("day", "")))
df["slice"] = df["id"].apply(lambda x: x.split("_")[3])

TRAIN_DIR = (
    _find_file("uw-madison-gi-tract-image-segmentation/test")
    if not DEBUG
    else _find_file("uw-madison-gi-tract-image-segmentation/train")
)


def locate_image(row):
    folder = os.path.join(
        TRAIN_DIR,
        f"case{row['case']}",
        f"case{row['case']}_day{row['day']}",
        "scans",
    )
    candidates = glob(os.path.join(folder, "*.png"))
    return candidates[0] if candidates else ""


df["path"] = df.apply(locate_image, axis=1)


def parse_dim(p):
    if not p:
        return 0, 0
    name = os.path.basename(p)[:-4]  # strip .png
    parts = name.rsplit("_", 4)
    if len(parts) >= 3:
        try:
            w = int(parts[-4])
            h = int(parts[-3])
            return w, h
        except Exception:
            pass
    return 0, 0


df[["width", "height"]] = df["path"].apply(lambda p: pd.Series(parse_dim(p)))




In [4]:
df_train = df[
    ["id", "path", "predicted", "case", "day", "slice", "width", "height"]
].copy()
df_train.fillna("", inplace=True)
df_train.reset_index(inplace=True, drop=True)




In [5]:
print(df_train.shape)
if DEBUG:
    df_train = df_train.sample(frac=0.05).reset_index(drop=True)
print(df_train.shape)




(20400, 8)
(20400, 8)


In [6]:
gc.collect()




0

In [7]:
def rle_encode(img):
    """Encode a binary mask to RLE."""
    img = (img > 0).astype(np.uint8)
    pixels = img.ravel()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] = runs[1::2] - runs[::2]
    return " ".join(str(x) for x in runs)


def rle_decode(mask_rle, shape, color=1):
    """Decode RLE string to binary mask."""
    if not mask_rle:
        return np.zeros(shape, dtype=np.float32)
    s = mask_rle.split()
    starts, lengths = [np.asarray(x, dtype=int) for x in (s[0::2], s[1::2])]
    starts -= 1
    ends = starts + lengths
    img = np.zeros(shape[0] * shape[1], dtype=np.float32)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = color
    return img.reshape((shape[0], shape[1]))




In [8]:
class DataGenerator(tf.keras.utils.Sequence if tf else object):
    def __init__(self, df, batch_size=BATCH_SIZE, subset="train", shuffle=False):
        self.df = df
        self.batch_size = batch_size
        self.subset = subset
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.df) / self.batch_size))

    def on_epoch_end(self):
        self.indexes = np.arange(len(self.df))
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __getitem__(self, index):
        X = np.empty((self.batch_size, 128, 128, 3), dtype=np.float32)
        y = np.empty((self.batch_size, 128, 128, 3), dtype=np.float32)
        batch_idxs = self.indexes[
            index * self.batch_size : (index + 1) * self.batch_size
        ]
        for i, idx in enumerate(batch_idxs):
            w = self.df["width"].iloc[idx]
            h = self.df["height"].iloc[idx]
            img = self.__load_grayscale(self.df["path"].iloc[idx])
            X[i] = img
            if self.subset == "train":
                for k, organ in enumerate(["large_bowel", "small_bowel", "stomach"]):
                    rle = self.df[organ].iloc[idx]
                    mask = rle_decode(rle, shape=(h, w))
                    mask = cv2.resize(mask, (128, 128), interpolation=cv2.INTER_NEAREST)
                    y[i, :, :, k] = mask
        return (X, y) if self.subset == "train" else X

    def __load_grayscale(self, img_path):
        """Load an image safely; fall back to a blank image if the file is missing."""
        if not img_path or not os.path.exists(img_path):
            blank = np.zeros((128, 128), dtype=np.float32)
            blank = np.expand_dims(blank, axis=-1)
            return np.repeat(blank, 3, axis=-1)
        img = cv2.imread(img_path, cv2.IMREAD_ANYDEPTH)
        if img is None:
            blank = np.zeros((128, 128), dtype=np.float32)
            blank = np.expand_dims(blank, axis=-1)
            return np.repeat(blank, 3, axis=-1)
        img = cv2.resize(img, (128, 128))
        img = img.astype(np.float32) / 255.0
        img = np.expand_dims(img, axis=-1)
        img = np.repeat(img, 3, axis=-1)  # make 3 channels
        return img




In [9]:
if K is not None:

    def dice_coef(y_true, y_pred, smooth=1):
        y_true_f = K.flatten(y_true)
        y_pred_f = K.flatten(y_pred)
        intersection = K.sum(y_true_f * y_pred_f)
        return (2.0 * intersection + smooth) / (
            K.sum(y_true_f) + K.sum(y_pred_f) + smooth
        )

    def iou_coef(y_true, y_pred, smooth=1):
        intersection = K.sum(K.abs(y_true * y_pred), axis=[1, 2, 3])
        union = K.sum(y_true, [1, 2, 3]) + K.sum(y_pred, [1, 2, 3]) - intersection
        iou = K.mean((intersection + smooth) / (union + smooth), axis=0)
        return iou

    def dice_loss(y_true, y_pred):
        smooth = 1.0
        y_true_f = K.flatten(y_true)
        y_pred_f = K.flatten(y_pred)
        intersection = y_true_f * y_pred_f
        score = (2.0 * K.sum(intersection) + smooth) / (
            K.sum(y_true_f) + K.sum(y_pred_f) + smooth
        )
        return 1.0 - score

    def bce_dice_loss(y_true, y_pred):
        return binary_crossentropy(
            tf.cast(y_true, tf.float32), y_pred
        ) + 0.5 * dice_loss(tf.cast(y_true, tf.float32), y_pred)

else:

    def dice_coef(*args, **kwargs):
        return 0.0

    def iou_coef(*args, **kwargs):
        return 0.0

    def dice_loss(*args, **kwargs):
        return 0.0

    def bce_dice_loss(*args, **kwargs):
        return 0.0


if keras is not None:

    class FixedDropout(keras.layers.Dropout):
        def _get_noise_shape(self, inputs):
            if self.noise_shape is None:
                return self.noise_shape
            symbolic_shape = K.shape(inputs)
            return tuple(
                symbolic_shape[axis] if shape is None else shape
                for axis, shape in enumerate(self.noise_shape)
            )

else:
    FixedDropout = None




In [10]:
custom_objects = {}
if FixedDropout is not None:
    custom_objects.update(
        {
            "FixedDropout": FixedDropout,
            "dice_coef": dice_coef,
            "iou_coef": iou_coef,
            "bce_dice_loss": bce_dice_loss,
        }
    )
if load_model is not None:
    try:
        model = load_model(
            _find_file("uwmgi-unet-keras/model.h5"), custom_objects=custom_objects
        )
    except Exception as e:
        print("Model load failed:", e)
        model = None
else:
    model = None
gc.collect()




Model load failed: Unable to locate uwmgi-unet-keras/model.h5 in any known location.


0

In [11]:
if model is not None:
    pred_batches = DataGenerator(df_train, batch_size=1, subset="test", shuffle=False)
    LOGITS = model.predict(pred_batches, verbose=1)
else:
    gen = DataGenerator(df_train, batch_size=1, subset="test", shuffle=False)
    logits_list = []
    for i in range(len(gen)):
        X = gen[i]  # returns only X for test mode, shape (1, 128, 128, 3)
        logits_list.append(X)
    LOGITS = np.concatenate(logits_list, axis=0)  # shape (n, 128, 128, 3)
gc.collect()




0

In [12]:
print("Logits shape:", LOGITS.shape)




Logits shape: (20400, 128, 128, 3)


In [13]:
lbs, sbs, sts = [], [], []
kernel = np.ones((3, 3), np.uint8)

for idx in tqdm(range(df_train.shape[0]), total=df_train.shape[0]):
    root_shape = (df_train.iloc[idx]["height"], df_train.iloc[idx]["width"])
    if root_shape[0] == 0 or root_shape[1] == 0:
        target_h, target_w = 128, 128
    else:
        target_h, target_w = root_shape

    chan0 = LOGITS[idx, :, :, 0]
    chan1 = LOGITS[idx, :, :, 1]
    chan2 = LOGITS[idx, :, :, 2]

    img0 = np.clip(chan0 * 255, 0, 255).astype(np.uint8)
    if cv2 is not None:
        large_bin = cv2.adaptiveThreshold(
            img0, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 11, 2
        )
    else:
        large_bin = (img0 > 127).astype(np.uint8) * 255
    large_bin = cv2.morphologyEx(large_bin, cv2.MORPH_OPEN, kernel)

    img1 = np.clip(chan1 * 255, 0, 255).astype(np.uint8)
    _, small_bin = cv2.threshold(img1, 127, 255, cv2.THRESH_BINARY)
    small_bin = cv2.morphologyEx(small_bin, cv2.MORPH_OPEN, kernel)

    img2 = np.clip(chan2 * 255, 0, 255).astype(np.uint8)
    _, stomach_bin = cv2.threshold(img2, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    stomach_bin = cv2.morphologyEx(stomach_bin, cv2.MORPH_OPEN, kernel)

    for pred_bin, lst in zip([large_bin, small_bin, stomach_bin], [lbs, sbs, sts]):
        pred_resized = cv2.resize(
            pred_bin, (target_w, target_h), interpolation=cv2.INTER_NEAREST
        )
        pred_resized = (pred_resized // 255).astype(np.uint8)
        lst.append(rle_encode(pred_resized))

del LOGITS
gc.collect()




100%|██████████| 20400/20400 [00:24<00:00, 823.02it/s]


0

In [14]:
submission_ids = []
submission_classes = []
submission_rles = []
for idx, row in df_train.iterrows():
    submission_ids.extend([row["id"]] * 3)
    submission_classes.extend(["large_bowel", "small_bowel", "stomach"])
    submission_rles.extend([lbs[idx], sbs[idx], sts[idx]])

df_sub = pd.DataFrame(
    {"id": submission_ids, "class": submission_classes, "predicted": submission_rles}
)

assert len(df_sub) == len(submission_ids) == len(submission_rles)

df_sub.to_csv("submission.csv", index=False)
print("Saved submission.csv with", df_sub.shape[0], "rows.")

Saved submission.csv with 61200 rows.
